# Stage 1 — Industry input–output accounts from 2023 supply and use tables

This notebook transforms the public TurkStat supply and use matrices using the fixed product-sales structure (Model D). Monetary flows are in thousand TRY. Product and industry identifiers preserve the source ordering.

## 1. Libraries and portable paths

Input and output directories are resolved relative to this notebook. No personal filesystem paths are required.

In [ ]:
ANALYSIS = 'analysis'
PREFIX = 'I'

from pathlib import Path
import json, hashlib, tempfile, subprocess, shutil, os
import numpy as np
import pandas as pd
from IPython.display import display

def locate_analysis(name):
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if parent.name == name and (parent / 'datas').is_dir():
            return parent
        if (parent / name / 'datas').is_dir():
            return parent / name
    raise FileNotFoundError(f'{name} not found. Open the notebook from its codes directory.')

BASE = locate_analysis(ANALYSIS)
DATA, OUT = BASE / 'datas', BASE / 'outputs'
OUT.mkdir(exist_ok=True)
PRODUCTS = [f'pro{i}' for i in range(1, 65)]
INDUSTRIES = [f'ind{i}' for i in range(1, 65)]
CODES = list(range(64))
FINAL = ['Household', 'Government', 'Investment', 'Exports']
def read_input(name):
    return pd.read_excel(DATA / f'{PREFIX}_{name}.xlsx', index_col=0)
print('Analysis:', ANALYSIS, '| Reference year: 2023 | Monetary unit: thousand TRY')

## 2. Read and verify inputs

The input manifest checks the translated public workbooks. Dimensions, unique identifiers and complete product/industry coverage are verified before calculation.

In [ ]:
supply = read_input('supply_2023')
domestic = read_input('domestic_use_2023')
imports = read_input('import_use_2023')
labels = read_input('product_industry_mapping')
for frame in [supply, domestic, imports]:
    assert frame.index.is_unique and frame.columns.is_unique
    assert all(p in frame.index for p in PRODUCTS)
    assert all(i in frame.columns for i in INDUSTRIES)
manifest = json.loads((DATA / f'{PREFIX}_input_manifest.json').read_text(encoding='utf-8'))
for entry in manifest:
    path = DATA / entry['file']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry['sha256'], path.name
display(pd.DataFrame({'Table':['Supply','Domestic use','Import use'],
                      'Rows':[len(supply),len(domestic),len(imports)],
                      'Columns':[supply.shape[1],domestic.shape[1],imports.shape[1]]}))

## 3. Separate the economic blocks

Supply determines product and industry output. Domestic and imported use are separated into intermediate and four final-demand blocks; compensation, operating surplus and taxes retain their source values.

In [ ]:
G = supply.loc[PRODUCTS, INDUSTRIES].to_numpy(dtype=float)  
x = supply.loc[PRODUCTS, 'Total output'].to_numpy(dtype=float)  
g = supply.loc['Total output', INDUSTRIES].to_numpy(dtype=float)  
Ud = domestic.loc[PRODUCTS, INDUSTRIES].to_numpy(dtype=float)
Um = imports.loc[PRODUCTS, INDUSTRIES].to_numpy(dtype=float)
Yd = domestic.loc[PRODUCTS, 'Household':'Exports'].to_numpy(dtype=float)
Ym = imports.loc[PRODUCTS, 'Household':'Exports'].to_numpy(dtype=float)
W = domestic.loc['Compensation of employees':'Gross operating surplus', INDUSTRIES].to_numpy(dtype=float)
tax = domestic.loc['Net product taxes', INDUSTRIES].to_numpy(dtype=float)
tax_final = domestic.loc['Net product taxes', 'Household':'Exports'].to_numpy(dtype=float)
assert G.shape == Ud.shape == Um.shape == (64,64)
assert Yd.shape == Ym.shape == (64,4)
assert W.shape == (3,64)
assert all(np.isfinite(a).all() for a in [G,x,g,Ud,Um,Yd,Ym,W,tax,tax_final])
assert (x > 0).all() and (g > 0).all()
assert np.allclose(G.sum(axis=1),x) and np.allclose(G.sum(axis=0),g)
display(pd.DataFrame({'Measure':['Domestic output','Domestic intermediate use','Imported intermediate use'],
                      'Thousand TRY':[g.sum(),Ud.sum(),Um.sum()]}))

## 4. Model D transformation

The industry market-share matrix is $D=G^T\operatorname{diag}(x)^{-1}$. Applying it to product uses gives industry domestic/imported intermediate and final-demand matrices. Value added is already classified by industry.

In [ ]:
V = G.T
D = V @ np.linalg.inv(np.diag(x))
T = D
Bd = T @ Ud
Bm = T @ Um
Fd = T @ Yd
Fm = T @ Ym
W_sector = W.copy()  
ms = Bm.sum(axis=1) + Fm.sum(axis=1)
q = g + ms  

## 5. Accounting checks

Market shares sum to one. Product and industry balances and imported-use totals are tested independently. Signed investment and inventory entries are retained.

In [ ]:
checks = []
def check(name, actual, expected, atol=1e-4, rtol=1e-10):
    actual, expected = np.asarray(actual, dtype=float), np.asarray(expected, dtype=float)
    error = float(np.max(np.abs(actual - expected)))
    ok = bool(np.allclose(actual, expected, atol=atol, rtol=rtol))
    checks.append({'Check':name, 'Maximum absolute difference':error, 'Passed':ok})
    assert ok, f'{name}: {error}'

check('Market-share column sums', T.sum(axis=0), np.ones(64))
check('Domestic product balance', Ud.sum(axis=1)+Yd.sum(axis=1), x)
check('Domestic industry sales = output', Bd.sum(axis=1)+Fd.sum(axis=1), g)
check('Domestic industry costs = output', Bd.sum(axis=0)+Bm.sum(axis=0)+tax+W.sum(axis=0), g)
check('Imported intermediate use preserved', Bm.sum(axis=0), Um.sum(axis=0))
check('Imported final use preserved', Fm.sum(axis=0), Ym.sum(axis=0))
check('Total sales = output + imports', (Bd+Bm).sum(axis=1)+(Fd+Fm).sum(axis=1), q)
assert (T >= 0).all() and (Bd >= -1e-6).all() and (Bm >= -1e-6).all()

display(pd.DataFrame(checks))

## 6. Complete industry IO tables

The intermediate blocks are combined with final demand, factor payments and tax rows. The total-supply presentation adds an import-balancing row without counting imports twice as intermediate inputs.

In [ ]:
def io_table(Z, F, mode):
    cols = CODES + ['Subtotal'] + FINAL + ['Row total']
    table = pd.DataFrame(np.column_stack([Z,Z.sum(axis=1),F,Z.sum(axis=1)+F.sum(axis=1)]),index=CODES,columns=cols)
    table.index.name = 'Sector code / account'
    table.loc['Total intermediate use'] = table.sum(axis=0)
    def lower(name, industry_values, final_values=None):
        row = np.full(len(cols), np.nan)
        row[:64] = industry_values
        row[64] = np.sum(industry_values)
        if final_values is not None:
            row[65:69] = final_values
            row[69] = np.sum(industry_values) + np.sum(final_values)
        table.loc[name] = row
    if mode == 'imports':
        return table
    if mode == 'domestic':
        lower('Imported intermediate inputs', Bm.sum(axis=0), Fm.sum(axis=0))
    lower('Net product taxes', tax, tax_final)
    for title, row in zip(['Compensation of employees','Other net production taxes','Gross operating surplus'], W_sector):
        lower(title,row)
    if mode == 'supply':
        lower('Imports / supply balancing', ms)
        lower('Total supply', q)
    else:
        lower('Domestic output', g)
    return table

domestic_io = io_table(Bd,Fd,'domestic')
import_io = io_table(Bm,Fm,'imports')
supply_io = io_table(Bd+Bm,Fd+Fm,'supply')


sector_mapping = labels.copy()
sector_mapping.index.name = 'Sector code (0-63)'
display(domestic_io.iloc[:5,:5])

## 7. Portable workbook export

XlsxWriter exports values and Python-computed formula caches. Workbooks contain the same numerical results as the calculation arrays.

In [ ]:
def matrix_frame(a, rows=CODES, cols=CODES, index_name='Sector code'):
    return pd.DataFrame(a,index=pd.Index(rows,name=index_name),columns=cols)

def sheet(name, frame, money=False, text=False, summary=False):
    frame=frame.copy()
    if money:
        frame.index.name=str(frame.index.name or 'Sector code')+' (thousand TRY)'
    frame=frame.reset_index()
    values=[list(frame.columns)]+frame.astype(object).where(pd.notna(frame),None).values.tolist()
    
    values=json.loads(json.dumps(values,ensure_ascii=False,default=lambda x:x.item()))
    return {'name':name,'values':values,'money':money,'text':text,'summary':summary}

jobs=[]
def workbook(filename, sheets):
    jobs.append({'path':str(OUT/f'{PREFIX}_{filename}.xlsx'),'variant':PREFIX,'sheets':sheets})

workbook('D_market_shares_2023',[sheet('D',matrix_frame(T,cols=PRODUCTS))])
for name,arr in [('Bd_domestic_intermediate',Bd),('Bm_imported_intermediate',Bm)]:
    workbook(name+'_2023',[sheet(name[:31],matrix_frame(arr),money=True)])
for name,arr in [('Fd_domestic_final_demand',Fd),('Fm_imported_final_demand',Fm)]:
    workbook(name+'_2023',[sheet(name[:31],matrix_frame(arr,cols=FINAL),money=True)])

def export_workbooks():
    import sys
    for parent in [Path.cwd(),*Path.cwd().parents]:
        if (parent/'tools/public_excel_export.py').is_file():
            sys.path.insert(0,str(parent/'tools'));break
    from public_excel_export import export_jobs
    export_jobs(jobs)


## 8. Combined industry IO workbook

The workbook contains domestic IO, import IO, total supply, sector correspondence and methodological notes. Stage 2 reads the total-supply view.

In [ ]:
notes=pd.DataFrame([('Unit','2023; monetary values in thousand TRY; coefficients dimensionless.'),('Transformation','Model D: D=G.T @ diag(x)^(-1); Bd=D@Ud; Bm=D@Um; Fd=D@Yd; Fm=D@Ym.'),('Source','UN (2018), Handbook on Supply, Use and Input-Output Tables; original TurkStat supply/use ordering retained.'),('Totals','Domestic_IO output is g; Total_Supply includes g plus imports.'),('Scope','64 industries and four final uses. Source CIF/FOB and residency adjustments are retained outside the transformation core.'),('Next stage','Stage 2 reads this workbook and preserves its factor and tax flows.')],columns=['Topic','Description']).set_index('Topic')
workbook('industry_io_tables_2023',[
    sheet('Domestic_IO',domestic_io,money=True),sheet('Import_IO',import_io,money=True),
    sheet('Total_Supply',supply_io,money=True),sheet('Sector_List',sector_mapping,text=True),sheet('Description',notes,text=True)])
export_workbooks()


## 9. Round-trip verification

Every sheet is reopened and compared cell by cell with the exported arrays. The execution log records numerical checks and dependency versions.

In [ ]:
for job in jobs:
    path=Path(job['path'])
    assert path.exists() and path.stat().st_size>1000
    excel=pd.ExcelFile(path)
    assert excel.sheet_names==[s['name'] for s in job['sheets']]
    for item in job['sheets']:
        loaded=pd.read_excel(path,sheet_name=item['name'],header=None)
        expected=item['values']
        assert loaded.shape==(len(expected),len(expected[0])), (path.name,item['name'],loaded.shape)
        for r,row in enumerate(expected):
            for c,value in enumerate(row):
                actual=loaded.iat[r,c]
                if value is None:
                    assert pd.isna(actual)
                elif isinstance(value,bool):
                    assert bool(actual)==value
                elif isinstance(value,(int,float)):
                    assert np.isclose(float(actual),value,rtol=1e-12,atol=1e-6)
                else:
                    assert str(actual)==str(value), (item['name'],r,c,value,actual)
run_log={'analysis':ANALYSIS,'year':2023,'unit':'thousand TRY',
         'numpy':np.__version__,'pandas':pd.__version__,
         'checks':checks,'outputs':[Path(job['path']).name for job in jobs]}
(OUT/f'{PREFIX}_execution_log.json').write_text(json.dumps(run_log,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'{len(jobs)} workbooks and all worksheets reopened and verified.')

## Interpretation

Domestic and imported uses remain distinct. Import balancing is an accounting presentation of total supply; production output retains its original definition. Subsequent SAM construction uses these preserved flows.